In [1]:
import torch
print(torch.cuda.is_available())  # Should return True
print(torch.cuda.get_device_name(0))  #

True
NVIDIA GeForce RTX 3080 Ti


In [2]:
import os
from dotenv import load_dotenv

model_id_base = "meta-llama/Meta-Llama-3.1-8B" # General model

load_dotenv()

# Access variables
token = os.getenv("hf_token")

In [3]:
from transformers import AutoTokenizer, LlamaForCausalLM, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
import torch
compute_dtype = getattr(torch, "float16")

bnb_config = BitsAndBytesConfig(
            load_in_4bit=True, # 8 bit uses too much memory
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=compute_dtype,
            bnb_4bit_use_double_quant=True
        )
model_id_base = "meta-llama/Meta-Llama-3.1-8B" # General model


# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id_base)

# Load the model with quantization and auto device mapping
model = AutoModelForCausalLM.from_pretrained(
    model_id_base,
    quantization_config=bnb_config,
    device_map="auto"  # Automatically uses GPU, with CPU fallback if necessary
)

c:\Users\Brayden Turner\Projects\chat_playground\chat_playground\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 4/4 [00:22<00:00,  5.54s/it]


In [ ]:
from datasets import load_dataset

# Load dataset in streaming mode to process line by line (saves memory)
dataset = load_dataset(
        "json",
        data_files={"train": "data/krusty_krab.jsonl"},
        split="train",
        streaming=True  # Enables streaming mode
    )
dataset

IterableDataset({
    features: ['index', 'text', 'date_utc', 'is_from_me', 'cache_has_attachments', 'message_id', 'reaction', 'mime_type', 'name', 'members'],
    n_shards: 1
})

In [ ]:
import json
from tqdm import tqdm

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # Use EOS token for padding

# Sliding window parameters
window_size = 5   # Number of messages per window
overlap = 2       # Number of overlapping messages between windows

output_file = "data/tokenized_output.json"

# Persistent buffer to accumulate messages across stream batches
persistent_buffer = []  

def sliding_window(chat_history):
    """
    Process chat history into sliding window chunks with overlap.
    This helps maintain conversation context over multiple messages.

    Args:
        chat_history (list): List of chat messages (JSON objects).
    
    Returns:
        list: List of overlapping chunks of messages, each formatted for tokenization.
    """
    
    global persistent_buffer  # Use a global buffer to persist across calls
    chunks = []
    
    for message in chat_history:
        # Format message as "<name>: <text>"
        speaker = message['name']
        text = message['text']
        persistent_buffer.append(f"{speaker}: {text})")
        
        # When enough messages accumulate, create a sliding window chunk
        if len(persistent_buffer) >= window_size:
            # Take the first 'window_size' messages from the buffer
            chunk = persistent_buffer[:window_size]
            chunks.append({"messages": chunk})
            
            # Keep the last 'overlap' messages to ensure continuity across windows
            persistent_buffer = persistent_buffer[-overlap:]
            
    return chunks


def process_and_tokenize_stream(dataset):
    """
    Stream the chat data from JSONL file, apply sliding window processing, and tokenize.
    Uses Hugging Face datasets streaming to prevent memory overload.
    """
    
    buffer = []  # To accumulate and batch-process sliding windows
    batch_size = 100  # Tokenize every 100 chunks to avoid memory overflow
    
    for i, data in tqdm(enumerate(dataset)):
        buffer.append(data)
        
        # Once buffer reaches batch size, apply sliding window
        if len(buffer) >= batch_size:
            all_chunks = []
            
            # Process each conversation/message batch in sliding windows
            for chat in buffer:
                chunks = sliding_window([chat])
                all_chunks.extend(chunks)
            
            # Tokenize and save in batches
            tokenize_and_save(all_chunks)
            buffer = []  # Clear buffer after processing

    # Handle remaining buffer
    if buffer:
        all_chunks = []
        for chat in buffer:
            chunks = sliding_window([chat])
            all_chunks.extend(chunks)
        tokenize_and_save(all_chunks)


def tokenize_and_save(chunks):
    """
    Tokenize overlapping chat chunks and save the results to disk in JSON format.
    Handles Hugging Face tokenizer's BatchEncoding object by converting tensors to lists.

    Args:
        chunks (list): List of chat message chunks to be tokenized.
        output_file (str): Path to save the tokenized output.
    """
    tokenized_data = []
    
    for chunk in chunks:
        # Join chunk messages into a single block of text for tokenization
        conversation = "\n".join(chunk['messages'])
        
        # Tokenize the conversation (returns BatchEncoding object)
        tokens = tokenizer(
            conversation,
            truncation=True,
            padding="max_length",
            max_length=512,
            return_tensors="pt"
        )
        
         # Convert BatchEncoding (tensor) to Python lists for JSON serialization
        tokens_dict = {
            key: val.cpu().tolist()  # Convert tensors to lists
            for key, val in tokens.items()
        }
        
        tokenized_data.append(tokens_dict)

    # Save tokenized data to disk or use it directly
    with open(output_file, 'a') as f:
        for entry in tokenized_data:
            json.dump(entry, f)
            f.write('\n')



process_and_tokenize_stream(dataset)


In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./fine-tuned-group-chat",
    per_device_train_batch_size=4,
    num_train_epochs=3,
    save_steps=1000,
    learning_rate=5e-5,
    logging_steps=100,
    weight_decay=0.01,
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()